# The NormData class

A key component of the PCNtoolkit is the NormData object. It is a container for the data that will be used to fit the normative model. The NormData object keeps track of the all the dimensions of your data, the features and response variables, batch effects, preprocessing steps, and more. 

In [ ]:
import copy

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from pcntoolkit import NormData

## Creating a NormData object

There are currently two easy ways to create a NormData object. 
1. Load from a pandas dataframe
2. Load from numpy arrays 

Here are examples of both. 


In [ ]:
# Creating a NormData object from a pandas dataframe

# Download an example dataset:
data = pd.read_csv(
    "https://raw.githubusercontent.com/predictive-clinical-neuroscience/PCNtoolkit-demo/refs/heads/main/data/fcon1000.csv"
)

# specify the column names to use
covariates = ["age"]
batch_effects = ["sex", "site"]
response_vars = ["WM-hypointensities", "Left-Lateral-Ventricle", "Brain-Stem"]

# create a NormData object
norm_data = NormData.from_dataframe(
    name="fcon1000",
    dataframe=data,
    covariates=covariates,
    batch_effects=batch_effects,
    response_vars=response_vars,
    remove_outliers=True,
    z_threshold=10,
)
norm_data.coords

In [ ]:
# Creating a NormData object from numpy arrays
import numpy as np

from pcntoolkit import NormData

# synthesize some data
X = np.random.randn(100, 10)
Y = np.random.randn(100, 10)
batch_effects = np.random.randint(0, 2, 100)[:,None]
subject_ids = np.arange(100)

# Create a NormData object
np_norm_data = NormData.from_ndarrays("fcon1000", X=X, Y=Y, batch_effects=batch_effects, subject_ids=subject_ids)
np_norm_data.coords

As you can see, it is very simple to create a NormData object. 

There is an important difference though: the coordinates of the NormData object that was created with `from_dataframe` have the name of the column in the dataframe, but the `from_ndarrays` method creates coordinates with generic names. This is why the from_dataframe method is favorable.


## Casting back to a pandas dataframe

The NormData object can be cast back to a pandas dataframe using the `to_dataframe` method. This will return a pandas dataframe with a columnar multi-index. 


In [ ]:
df = norm_data.to_dataframe()
df.head()

## Inspecting the NormData 

So let's go over the attributes of the NormData object. Because it is a subclass of xarray.Dataset, it has all the attributes of a xarray.Dataset, but it has some additional attributes that are specific to normative modelling. 

### The data variables

The data variables of the NormData object are:
- `X`: The covariates
- `Y`: The response variables
- `batch_effects`: The batch effects
- `subjects`: The subject ids

And all these data variables are xarray.DataArrays, with corresponding dimensions, stored in the `data_vars` attribute of the NormData object.

In [ ]:
norm_data.data_vars

### The coordinates

Because it is a subclass of xarray.Dataset, the NormData object also holds all the coordinates of the data, found under the `coords` attribute. 

The coordinates are:
- `observations`: The index of the observations
- `response_vars`: The names of the response variables
- `covariates`: The names of the covariates
- `batch_effect_dims`: The names of the batch effect dimensions

In [ ]:
norm_data.coords

### Indexing using the coordinates

Xarrays powerful indexing methods can also be used on NormData. 

#### Selecting a response variable

For example, to select the data for a specific response variable, you can use the `response_vars` coordinate:

```python
norm_data.sel(response_vars="WM-hypointensities")
```

This will return a new NormData object with only the data for the response variable "WM-hypointensities".

In [ ]:
norm_data.sel(response_vars="WM-hypointensities")

#### Selecting a number of observations

But we can also filter out a slice of the data. For example, to select the first 10 observations, you can use the `observations` coordinate:

```python
norm_data.sel(observations=slice(0, 9))
```

This will return a new NormData object with only the first 10 observations.

In [ ]:
norm_data.sel(observations=slice(0, 9))

## NormData with predictions

After fitting a model and predicting on NormData, the NormData object will have new attributes holding the predictions. 


Specifically, the NormData object will be extended with new data variables:

- `Z`: The predicted Z scores for each response variable
- `centiles`: The predicted centiles
- `logp`: The predicted log-p-values for each response variable
- `Yhat`: The predicted mean of the response variable
- `Y_harmonized`: The harmonized response variables
- `statistics`: An array of statistics for each response variable


And the following new coordinates:
- `centile`: The specific centile values
- `statistic`: The name of the computed statistics


In [ ]:
from pcntoolkit import BLR, NormativeModel

# We create a very simple BLR model because it is fast to fit
model = NormativeModel(BLR())
model.fit(norm_data)  # Fitting on the data also makes predictions for that data

In [ ]:
norm_data.data_vars

In [ ]:
norm_data.coords

### Indexing of predicted data

All the indexing methods can still be used, and they will also slice through the newly added data variables. So for example, to select the first 10 observations, you can use:

```python
norm_data.sel(observations=slice(0, 9))
```

This will return a new NormData object with only the first 10 observations.

In [ ]:
norm_data.sel(observations=slice(0, 9))

Or, if we want to select only the WM-hypointensities, we can use:

```python
norm_data.sel(response_vars="WM-hypointensities")
```

This will return a new NormData object with only the WM-hypointensities.

In [ ]:
norm_data.sel(response_vars="WM-hypointensities").statistics

Now we can use the to_dataframe method to cast that selection back to a pandas dataframe.

In [ ]:
new_df = norm_data.sel(response_vars="WM-hypointensities").to_dataframe()
new_df.head()

This should give you a pretty good overview of how to work with NormData. Most of the functionality is built on top of xarray, so if you want to learn more about xarray, you can check out the [xarray documentation](https://docs.xarray.dev/en/stable/). However, the Xarray.DataSet class does not officially support being extended, so the API does not work completely as expected. 

If you have any suggestions for improvements, please let us know!



## Pre-processing and split datasets

Sometimes we have a dataset that is pre-split into train and test, and we want to use that exact data split to fit the model. We can then load the data into two NormData objects, but we have to make sure that the two datasets are compatible. This will ensure that the fitted model is applicable to both of them.   

In [ ]:
# Download an example dataset:
data = pd.read_csv(
    "https://raw.githubusercontent.com/predictive-clinical-neuroscience/PCNtoolkit-demo/refs/heads/main/data/fcon1000.csv"
)
# Create an arbitrary split as a placeholder for a predefined split ()
train, test = train_test_split(data, test_size=100)

In [ ]:
# specify the column names to use
covariates = ["age"]
batch_effects = ["sex", "site"]
response_vars = ["WM-hypointensities", "Left-Lateral-Ventricle", "Brain-Stem"]

# create NormData objects
norm_train = NormData.from_dataframe(
    name="train", dataframe=train, covariates=covariates, batch_effects=batch_effects, response_vars=response_vars
)
norm_test = NormData.from_dataframe(
    name="test", dataframe=test, covariates=covariates, batch_effects=batch_effects, response_vars=response_vars
)

In [ ]:
# Should print false, because the train and test split do not contain the same sites
print(norm_train.check_compatibility(norm_test))

In [ ]:
norm_train.check_compatibility(norm_test)

In [ ]:
norm_train.make_compatible(norm_test)

In [ ]:
norm_train.check_compatibility(norm_test)